# Debug: model mistakes on val set
Run **Cell 1** once to load everything, then re-run **Cell 2** to sample a random mistake.

In [ ]:
import random
import sys
import torch
import torch.nn.functional as F
from pathlib import Path

sys.path.insert(0, str(Path().resolve()))

from main import DATA_ROOT, TOKENIZER, DECODER, CKPT_PATH, build_model, build_tokenizer
from data_processor.data import create_dataloaders
from data_processor.postprocessor import RussianToDigitLevenshtein

CKPT   = CKPT_PATH   # override here if needed, e.g. Path('logs/best_model.pth')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

_, val_loader, tokenizer = create_dataloaders(
    data_root_train=DATA_ROOT,
    data_root_dev=DATA_ROOT,
    batch_size=32,
    num_workers=0,
    tokenizer_type=TOKENIZER,
    augment_train=False,
)

model = build_model(tokenizer, DECODER).to(DEVICE)
model.load_state_dict(torch.load(CKPT, map_location=DEVICE, weights_only=True))
model.eval()

_to_digits = RussianToDigitLevenshtein()
BLANK_ID   = tokenizer.pad_id

def token_name(tid: int) -> str:
    return '<blank>' if tid == BLANK_ID else repr(tokenizer.id2token[tid])

# ── collect all val mistakes ──────────────────────────────────────────────────
mistakes = []

with torch.no_grad():
    for batch in val_loader:
        features        = batch['features'].to(DEVICE)
        feature_lengths = batch['feature_lengths']
        encoder_lengths = model.get_encoder_lengths(feature_lengths)

        logits    = model(features)
        log_probs = F.log_softmax(logits, dim=-1)
        raw_ids   = logits.argmax(dim=-1)

        decoded_tokens = model.decode(log_probs, encoder_lengths)

        for i, (tokens, ref, filename) in enumerate(
            zip(decoded_tokens, batch['transcriptions'], batch['filenames'])
        ):
            russian_text = tokenizer.join(tokens)
            pred_digits  = _to_digits.convert(russian_text)

            if pred_digits == ref:
                continue

            T       = int(encoder_lengths[i].item())
            raw_seq = raw_ids[i, :T].cpu().tolist()

            collapsed = []
            prev = None
            for tid in raw_seq:
                if tid != prev:
                    collapsed.append(tid)
                prev = tid

            mistakes.append({
                'filename':           filename,
                'ref':                ref,
                'pred_digits':        pred_digits,
                'russian_text':       russian_text,
                'raw_seq':            raw_seq,
                'collapsed':          collapsed,
                'collapsed_no_blank': [t for t in collapsed if t != BLANK_ID],
            })

print(f'Total val mistakes: {len(mistakes)} / {len(val_loader.dataset)}')


In [8]:
# ── re-run this cell to inspect a new random mistake ────────────────────────
m = random.choice(mistakes)

print('FILE :', m['filename'])
print('REF  :', m['ref'])
print('PRED :', m['pred_digits'])
print()

# 1. Raw per-frame argmax (every encoder frame, including blanks and repeats)
raw_named = [token_name(tid) for tid in m['raw_seq']]
print('── RAW FRAMES (argmax, no collapse) ──────────────────────────────')
# Print as a compact sequence; group runs of the same token to save space
runs = []
for name in raw_named:
    if runs and runs[-1][0] == name:
        runs[-1][1] += 1
    else:
        runs.append([name, 1])
print(' | '.join(f'{n}×{c}' if c > 1 else n for n, c in runs))
print()

# 2. After CTC collapse (remove consecutive duplicates, keep blanks)
print('── AFTER REMOVING CONSECUTIVE DUPLICATES ─────────────────────────')
print(' '.join(token_name(tid) for tid in m['collapsed']))
print()

# 3. After removing blanks → Russian words
print('── AFTER REMOVING BLANKS (decoded tokens → Russian) ──────────────')
decoded_tokens_named = [token_name(tid) for tid in m['collapsed_no_blank']]
print(' '.join(decoded_tokens_named))
print('Russian text:', m['russian_text'] if m['russian_text'] else '(empty)')
print()

# 4. Final digit conversion
print('── DIGIT CONVERSION ──────────────────────────────────────────────')
print(f'Russian → digits : "{m["russian_text"]}" → "{m["pred_digits"]}"')
print(f'Ground truth     : "{m["ref"]}"')

FILE : dev/a56a7d62f8.mp3
REF  : 331401
PRED : 331060

── RAW FRAMES (argmax, no collapse) ──────────────────────────────
<blank> | 'т'×2 | 'р'×2 | 'и' | 'с'×2 | 'т'×2 | 'а' | '<space>'×2 | 'т' | 'р'×2 | 'и' | 'д' | 'ц' | 'а' | 'т'×2 | 'ь' | '<space>'×2 | 'о'×2 | 'д' | 'н'×2 | 'а' | '<space>'×2 | 'т'×2 | 'ы'×2 | 'с' | 'я' | 'ч'×2 | '<space>'×2 | <blank>×6 | 'ш' | 'е' | 'с'×2 | 'т' | 'ь' | 'д' | 'е'×2 | 'с' | 'я' | 'т'×2 | '<space>'×2 | <blank> | 'о' | 'д' | 'и'

── AFTER REMOVING CONSECUTIVE DUPLICATES ─────────────────────────
<blank> 'т' 'р' 'и' 'с' 'т' 'а' '<space>' 'т' 'р' 'и' 'д' 'ц' 'а' 'т' 'ь' '<space>' 'о' 'д' 'н' 'а' '<space>' 'т' 'ы' 'с' 'я' 'ч' '<space>' <blank> 'ш' 'е' 'с' 'т' 'ь' 'д' 'е' 'с' 'я' 'т' '<space>' <blank> 'о' 'д' 'и'

── AFTER REMOVING BLANKS (decoded tokens → Russian) ──────────────
'т' 'р' 'и' 'с' 'т' 'а' '<space>' 'т' 'р' 'и' 'д' 'ц' 'а' 'т' 'ь' '<space>' 'о' 'д' 'н' 'а' '<space>' 'т' 'ы' 'с' 'я' 'ч' '<space>' 'ш' 'е' 'с' 'т' 'ь' 'д' 'е' 'с' 'я' 'т' '<space>